In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

In [5]:
df = pd.read_csv("../data/data.csv")
encoding="cp949"

In [6]:
df.head()

,출발항코드(DEPARTURE_PORT_CODE),출발항명(DEPARTURE_PORT_NAME),도착항코드(DEST_PORT_CODE),도착항명(DEST_PORT_NAME),도착항국가(DEST_COUNTRY),기준일자(DATE),항만효율성(PORT_EFFICIENCY),총항해시간(TOTAL_SAILING_TIME),대기시간(WAITING_TIME),항만정시성(ON_TIME_PERFORMANCE)
0,KRPUS,부산항,AEJEA,제벨알리항,AE,2025-05-28,23.69,644.50,0.0,7.94
1,KRPUS,부산항,AEJEA,제벨알리항,AE,2025-06-11,22.01,608.54,0.0,16.20
2,KRPUS,부산항,AEJEA,제벨알리항,AE,2025-06-11,16.52,749.45,0.0,1.36
3,KRPUS,부산항,AEJEA,제벨알리항,AE,2025-06-30,22.37,634.28,0.0,7.44
4,KRPUS,부산항,AEJEA,제벨알리항,AE,2025-06-30,16.37,654.99,0.0,1.80


In [7]:
df.shape

(480, 10)

In [8]:
df.columns

Index(['출발항코드(DEPARTURE_PORT_CODE)', '출발항명(DEPARTURE_PORT_NAME)',
       '도착항코드(DEST_PORT_CODE)', '도착항명(DEST_PORT_NAME)', '도착항국가(DEST_COUNTRY)',
       '기준일자(DATE)', '항만효율성(PORT_EFFICIENCY)', '총항해시간(TOTAL_SAILING_TIME)',
       '대기시간(WAITING_TIME)', '항만정시성(ON_TIME_PERFORMANCE)'],
      dtype='str')

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   출발항코드(DEPARTURE_PORT_CODE)  480 non-null    str    
 1   출발항명(DEPARTURE_PORT_NAME)   480 non-null    str    
 2   도착항코드(DEST_PORT_CODE)       480 non-null    str    
 3   도착항명(DEST_PORT_NAME)        480 non-null    str    
 4   도착항국가(DEST_COUNTRY)         426 non-null    str    
 5   기준일자(DATE)                  480 non-null    str    
 6   항만효율성(PORT_EFFICIENCY)      480 non-null    float64
 7   총항해시간(TOTAL_SAILING_TIME)   480 non-null    float64
 8   대기시간(WAITING_TIME)          480 non-null    float64
 9   항만정시성(ON_TIME_PERFORMANCE)  480 non-null    float64
dtypes: float64(4), str(6)
memory usage: 59.8 KB


In [10]:
df.describe()

,항만효율성(PORT_EFFICIENCY),총항해시간(TOTAL_SAILING_TIME),대기시간(WAITING_TIME),항만정시성(ON_TIME_PERFORMANCE)
count,480.000000,480.000000,480.000000,480.000000
mean,19.756458,432.292521,0.138042,6.431250
std,5.754084,415.966180,0.269674,25.602098
min,6.950000,28.970000,0.000000,-322.050000
25%,16.350000,75.985000,0.000000,1.940000
50%,18.735000,327.210000,0.030000,7.645000
75%,21.427500,624.942500,0.170000,13.950000
max,47.010000,1632.540000,3.040000,135.920000


### 기술통계 확인 결과
- 총항해시간은 평균 약 432.29시간, 중앙값 약 327.21시간, 최소 28.97시간에서 최대 1,632.54시간까지 큰 차이를 보임 >> 목적지별 항해거리 차이 포함되어 있으므로
- 따라서, 각 노선의 항해시간 분포를 기준으로 이상치를 탐지하는 것이 적절하다.

In [11]:
df.isna().sum()

출발항코드(DEPARTURE_PORT_CODE)     0
출발항명(DEPARTURE_PORT_NAME)      0
도착항코드(DEST_PORT_CODE)          0
도착항명(DEST_PORT_NAME)           0
도착항국가(DEST_COUNTRY)           54
기준일자(DATE)                     0
항만효율성(PORT_EFFICIENCY)         0
총항해시간(TOTAL_SAILING_TIME)      0
대기시간(WAITING_TIME)             0
항만정시성(ON_TIME_PERFORMANCE)     0
dtype: int64

In [12]:
df['기준일자(DATE)'] = pd.to_datetime(df['기준일자(DATE)'])

df['기준일자(DATE)'].dtype

dtype('<M8[us]')

In [13]:
df['기준일자(DATE)'].min(), df['기준일자(DATE)'].max()

(Timestamp('2025-05-28 00:00:00'), Timestamp('2026-06-22 00:00:00'))

In [14]:
df['출발항명(DEPARTURE_PORT_NAME)'].value_counts()

출발항명(DEPARTURE_PORT_NAME)
부산항    480
Name: count, dtype: int64

In [15]:
df['도착항명(DEST_PORT_NAME)'].value_counts()

도착항명(DEST_PORT_NAME)
KOBC PORT&LOGISTICS INDICATOR (종합)    54
싱가포르항                                 54
상하이항                                  52
로스앤젤레스항                               52
호치민항                                  50
칭다오항                                  49
제벨알리항                                 37
뉴욕항                                   35
홍콩항                                   31
로테르담항                                 25
안트베르펜항                                23
함부르크항                                 15
램차방항                                   3
Name: count, dtype: int64

- 출발항에서 각 도착항으로 향하는 노선을 각각 분석하여 각 노선의 항해시간 분포를 이용해 이상치를 발견한다.

In [16]:
df[
    df['도착항명(DEST_PORT_NAME)'] == 'KOBC PORT&LOGISTICS INDICATOR (종합)'
]

,출발항코드(DEPARTURE_PORT_CODE),출발항명(DEPARTURE_PORT_NAME),도착항코드(DEST_PORT_CODE),도착항명(DEST_PORT_NAME),도착항국가(DEST_COUNTRY),기준일자(DATE),항만효율성(PORT_EFFICIENCY),총항해시간(TOTAL_SAILING_TIME),대기시간(WAITING_TIME),항만정시성(ON_TIME_PERFORMANCE)
207,KRPUS,부산항,KPLI,KOBC PORT&LOGISTICS INDICATOR (종합),NaN,2025-05-28,18.99,239.37,0.13,2.29
208,KRPUS,부산항,KPLI,KOBC PORT&LOGISTICS INDICATOR (종합),NaN,2025-06-11,21.48,280.38,0.06,10.84
209,KRPUS,부산항,KPLI,KOBC PORT&LOGISTICS INDICATOR (종합),NaN,2025-06-11,20.45,446.38,0.00,23.58
210,KRPUS,부산항,KPLI,KOBC PORT&LOGISTICS INDICATOR (종합),NaN,2025-06-30,20.06,324.43,0.19,5.72
211,KRPUS,부산항,KPLI,KOBC PORT&LOGISTICS INDICATOR (종합),NaN,2025-06-30,20.67,319.09,0.11,9.72
212,KRPUS,부산항,KPLI,KOBC PORT&LOGISTICS INDICATOR (종합),NaN,2025-06-30,19.70,299.25,0.07,13.01
213,KRPUS,부산항,KPLI,KOBC PORT&LOGISTICS INDICATOR (종합),NaN,2025-07-08,21.16,361.97,0.12,4.02
214,KRPUS,부산항,KPLI,KOBC PORT&LOGISTICS INDICATOR (종합),NaN,2025-07-16,19.06,251.70,0.11,0.90
215,KRPUS,부산항,KPLI,KOBC PORT&LOGISTICS INDICATOR (종합),NaN,2025-07-29,18.63,232.41,0.15,3.09
216,KRPUS,부산항,KPLI,KOBC PORT&LOGISTICS INDICATOR (종합),NaN,2025-07-29,19.45,326.72,0.07,7.87


- 도착항 목록에 실제 항만이 아닌 'KOBC PORT&LOGISTICS INDICATOR (종합)'이 포함되어 있어 제외하고 실제 도착항 데이터만으로 분석한다.

In [17]:
route_df = df[
    df['도착항명(DEST_PORT_NAME)'] != 'KOBC PORT&LOGISTICS INDICATOR (종합)'
].copy()

In [18]:
route_df['도착항명(DEST_PORT_NAME)'].unique()

<ArrowStringArray>
[  '제벨알리항',  '안트베르펜항',    '상하이항',    '칭다오항',   '함부르크항',     '홍콩항',   '로테르담항',
   '싱가포르항',    '램차방항', '로스앤젤레스항',     '뉴욕항',    '호치민항']
Length: 12, dtype: str

In [19]:
fig = px.box(
    route_df,
    x='도착항명(DEST_PORT_NAME)',
    y='총항해시간(TOTAL_SAILING_TIME)',
    points='outliers',
    title='노선별 총항해시간 이상치 분석'
)

fig.show()

### 노선별 총항해시간 분포 확인
- 노선별 Box Plot을 통해 각 노선의 항해시간 중앙값과 분포가 서로 다르게 나타나는 것을 확인하였지만, 목적지와 항로 차이로 인해 발생할 수 있으므로 이상치로 판단할 수 없다.
- 따라서 각 노선별로 이상치를 판단하여 분석한다.

In [20]:
col = '총항해시간(TOTAL_SAILING_TIME)'
route = '도착항명(DEST_PORT_NAME)'

q1 = route_df.groupby(route)[col].transform('quantile', 0.25)
q3 = route_df.groupby(route)[col].transform('quantile', 0.75)

iqr = q3 - q1

### 노선별 사분위수 계산

- 노선마다 항해시간이 다르기 때문에 노선별로 Q1과 Q3를 구했다.

- Q1과 Q3의 차이인 IQR을 이용하여 각 노선에서 평소 범위를 벗어난 항해시간을 찾고자 한다.

In [21]:
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

In [22]:
outliers = route_df[
    (route_df[col] < lower) |
    (route_df[col] > upper)
].copy()

### 노선별 총항해시간 이상치 추출

- 각 노선의 총항해시간이 IQR 기준으로 계산한 하한보다 작거나, 상한보다 큰 데이터를 이상치로 추출하였다.

- 따라서 여기서 탐지되는 이상치는 단순히 항해시간이 긴 노선이 아니라, 해당 노선의 일반적인 항해시간 분포에서 벗어난 관측치를 의미한다.

In [23]:
len(outliers)

14

In [24]:
outliers[route].value_counts()

도착항명(DEST_PORT_NAME)
싱가포르항      3
로스앤젤레스항    3
제벨알리항      2
칭다오항       2
뉴욕항        2
안트베르펜항     1
상하이항       1
Name: count, dtype: int64

In [25]:
outliers[
    [
        '기준일자(DATE)',
        '출발항명(DEPARTURE_PORT_NAME)',
        '도착항명(DEST_PORT_NAME)',
        '총항해시간(TOTAL_SAILING_TIME)'
    ]
]

,기준일자(DATE),출발항명(DEPARTURE_PORT_NAME),도착항명(DEST_PORT_NAME),총항해시간(TOTAL_SAILING_TIME)
2,2025-06-11,부산항,제벨알리항,749.45
19,2025-10-15,부산항,제벨알리항,453.14
50,2026-02-02,부산항,안트베르펜항,1632.54
80,2025-10-22,부산항,상하이항,29.14
133,2025-11-05,부산항,칭다오항,28.97
154,2026-04-27,부산항,칭다오항,51.79
305,2025-10-15,부산항,싱가포르항,487.31
311,2025-11-18,부산항,싱가포르항,469.02
336,2026-06-15,부산항,싱가포르항,284.15
368,2025-12-02,부산항,로스앤젤레스항,430.84


### 노선별 이상치 발생 현황
- 모든 노선에서 이상치가 발생된 것이 아니라, 일부 노선에서 여러건의 이상치가 발생된 것을 확인 할 수 있었다.
- 따라서 이후에는 이상치가 발생한 노선과 발생 시점을 확인하여 해당 시기의 항해시간이 평상시 분포에서 어느 정도 벗어났는지 살펴볼 필요가 있다.

In [26]:
route_df['하한'] = lower
route_df['상한'] = upper

In [27]:
outliers = route_df[
    (route_df[col] < route_df['하한']) |
    (route_df[col] > route_df['상한'])
].copy()

In [28]:
outliers[
    [
        '기준일자(DATE)',
        '도착항명(DEST_PORT_NAME)',
        '총항해시간(TOTAL_SAILING_TIME)',
        '하한',
        '상한'
    ]
]


,기준일자(DATE),도착항명(DEST_PORT_NAME),총항해시간(TOTAL_SAILING_TIME),하한,상한
2,2025-06-11,제벨알리항,749.45,547.64000,747.56000
19,2025-10-15,제벨알리항,453.14,547.64000,747.56000
50,2026-02-02,안트베르펜항,1632.54,1046.96500,1624.84500
80,2025-10-22,상하이항,29.14,29.95875,50.86875
133,2025-11-05,칭다오항,28.97,29.45000,50.49000
154,2026-04-27,칭다오항,51.79,29.45000,50.49000
305,2025-10-15,싱가포르항,487.31,286.07250,452.87250
311,2025-11-18,싱가포르항,469.02,286.07250,452.87250
336,2026-06-15,싱가포르항,284.15,286.07250,452.87250
368,2025-12-02,로스앤젤레스항,430.84,262.99875,429.56875


### 이상치와 정상 범위 비교

- 이상치로 확인된 항해시간을 각 노선의 하한과 상한 기준과 비교하였다.

- 이를 통해 이상치가 해당 노선의 일반적인 항해시간 범위에서 얼마나 벗어났는지 확인할 수 있다.